# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vishalseries7030/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 1. Unit of analysis + time window

The primary daily performance table has one row per report date, pseudonymized client, and pseudonymized content item. The expected composite key is `report_date`, `client_hash_id`, and `content_hash_id`.

The March 2026 partition contains 9,841,378 rows covering 1–31 March 2026. The composite key check returned zero duplicate-key rows for this partition.

The content dimension contains 519,606 rows, with one row per pseudonymized content item. The query-performance table contains 2,414,248 rows, one row per pseudonymized client, content item, and query hash, for a fixed 90-day window from 2 April to 30 June 2026.

These tables have different grains and must not be joined without checking key uniqueness and join cardinality. March 2026 is used as the mid-panel validation month. This descriptive validation does not by itself establish future performance or causal impact.

In [49]:

!pip -q install duckdb huggingface_hub pyarrow pandas scikit-learn
print("Setup complete")


Setup complete


In [50]:
from huggingface_hub import login
login()

In [51]:
from huggingface_hub import HfApi

api = HfApi()

try:
    files = api.list_repo_files(
        repo_id="FlyRank/internship-warehouse",
        repo_type="dataset"
    )
    print("Warehouse access successful!")
    print("Total files listed:", len(files))

    print("\nMarch 2026 partition:")
    for f in files:
        if "month=2026-03" in f:
            print(f)
except Exception as e:
    print("Error:", type(e).__name__, str(e)[:300])

Warehouse access successful!
Total files listed: 24

March 2026 partition:
fact_content_daily_performance/month=2026-03/data_0.parquet


In [52]:

# ML-04: Inspect the current notebook environment
import os
from pathlib import Path

print("Current directory:", os.getcwd())
print("\nAvailable files in current directory:")
for path in list(Path(".").iterdir())[:30]:
    print(path)


Current directory: /content

Available files in current directory:
.config
sample_data


In [53]:
import duckdb

print("DuckDB version:", duckdb.__version__)

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

print("httpfs loaded successfully")

DuckDB version: 1.3.2
httpfs loaded successfully


In [54]:
from huggingface_hub import hf_hub_download
from pathlib import Path

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet"
)

print("March file downloaded:", Path(march_path).exists())
print("File size (MB):", round(Path(march_path).stat().st_size / (1024 * 1024), 2))

March file downloaded: True
File size (MB): 118.48


In [55]:
import duckdb

con = duckdb.connect()

march_sql_path = march_path.replace("\\", "/").replace("'", "''")

query1 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date) AS distinct_dates,
    COUNT(DISTINCT client_hash_id) AS distinct_clients,
    COUNT(DISTINCT content_hash_id) AS distinct_content_items,
    COUNT(DISTINCT (
        CAST(report_date AS VARCHAR) || '|' ||
        client_hash_id || '|' ||
        content_hash_id
    )) AS distinct_daily_keys
FROM read_parquet('{march_sql_path}')
"""

In [56]:
print("March path exists:", __import__("os").path.exists(march_path))
print("March path:", march_path)

print(
    duckdb.connect().execute(
        f"SELECT COUNT(*) FROM read_parquet('{march_path}')"
    ).fetchone()
)

March path exists: True
March path: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet
(9841378,)


In [57]:
query1 = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date
FROM read_parquet('{march_path}')
"""

result1 = con.execute(query1).fetchdf()
print(result1.to_string(index=False))

 total_rows first_report_date last_report_date
    9841378        2026-03-01       2026-03-31


In [58]:
query2 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT report_date) AS distinct_dates,
    COUNT(DISTINCT client_hash_id) AS distinct_clients,
    COUNT(DISTINCT content_hash_id) AS distinct_content_items
FROM read_parquet('{march_path}')
"""

result2 = con.execute(query2).fetchdf()
print(result2.to_string(index=False))

 total_rows  distinct_dates  distinct_clients  distinct_content_items
    9841378              31                55                  331437


In [59]:
query3 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS rows_gsc_available,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS FALSE
    ) AS rows_gsc_unavailable,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS NULL
    ) AS rows_gsc_null
FROM read_parquet('{march_path}')
"""

result3 = con.execute(query3).fetchdf()
print(result3.to_string(index=False))

 total_rows  rows_gsc_available  rows_gsc_unavailable  rows_gsc_null
    9841378             3611061               6230317              0


In [60]:
query4 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT (
        CAST(report_date AS VARCHAR) || '|' ||
        client_hash_id || '|' ||
        content_hash_id
    )) AS duplicate_key_rows
FROM read_parquet('{march_path}')
"""

result4 = con.execute(query4).fetchdf()
print(result4.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  duplicate_key_rows
    9841378                   0


## 2. Fields: feature / label / context / excluded

**Candidate features:** Content metadata such as `content_type`, `search_volume`, `competition`, `main_intent`, `word_count`, `char_count`, `backlinks`, and `category_count`. These are candidates only; their availability at the prediction time and missingness must be checked before modeling.

**Label:** Next-day `gsc_clicks`, using the following report date for the same client-content pair. Only rows with an observed next-day record and `gsc_data_available IS TRUE` for that next-day outcome should be considered. The label must be shifted forward within each client-content pair, not taken from the same day's feature row.

**Context fields:** `report_date`, `month`, `client_has_gsc`, `client_has_ga4`, `gsc_data_available`, and `ga4_data_available`. These fields help define time and data coverage; availability flags should not automatically be treated as predictive features.

**Excluded fields:** Hashed client, content, keyword, URL, and query identifiers are for controlled joins and grouping, not direct predictive features. Same-day and future performance measures—including `gsc_clicks`, `gsc_impressions`, and `gsc_avg_position`—must not be used as features for predicting next-day clicks unless they are explicitly transformed into lagged values available at prediction time. Raw or identifying client information, domains, URLs, private queries, and credentials must not appear in public outputs.

Missing performance values must be distinguished from genuine zero activity using the relevant availability flags. Join cardinality, feature timestamps, and possible leakage must be checked before modeling.

In [61]:

from huggingface_hub import HfApi

api = HfApi()

try:
    info = api.dataset_info("FlyRank/internship-warehouse")
    print("Dataset metadata accessible:", info.id)

    files = api.list_repo_files(
        "FlyRank/internship-warehouse",
        repo_type="dataset"
    )
    print("Warehouse files visible:", len(files))
except Exception as e:
    print(type(e).__name__, str(e)[:500])


Dataset metadata accessible: FlyRank/internship-warehouse
Warehouse files visible: 24


In [62]:

from huggingface_hub import login

login()
print("Hugging Face login successful!")

Hugging Face login successful!


## 3. Verify it with queries (grain, counts, missing values, windows)

**Query 1 — Grain and time window:** The March 2026 partition contains 9,841,378 rows covering 1–31 March 2026. The distinct daily-key count is also 9,841,378, so no duplicate composite keys were detected.

**Query 2 — GSC availability distribution:** Of the 9,841,378 rows, 3,611,061 have `gsc_data_available IS TRUE`, 6,230,317 have it set to `FALSE`, and zero have a `NULL` availability flag.

**Query 3 — Explicit availability filter:** Applying `WHERE gsc_data_available IS TRUE` retains 3,611,061 rows. This confirms that unavailable GSC performance must not be treated as zero activity.

These checks validate the selected partition's grain, date coverage, and GSC availability. They do not establish feature availability at prediction time or guarantee that a next-day target can be constructed for every row.

In [63]:
# Query 1: Verify March partition grain, row count, and date window

query1 = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_report_date,
    MAX(report_date) AS last_report_date,
    COUNT(DISTINCT (
        CAST(report_date AS VARCHAR) || '|' ||
        client_hash_id || '|' ||
        content_hash_id
    )) AS distinct_daily_keys
FROM read_parquet('{march_path}')
"""

result1 = con.execute(query1).fetchdf()
display(result1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,first_report_date,last_report_date,distinct_daily_keys
0,9841378,2026-03-01,2026-03-31,9841378


In [64]:
# Query 2: Verify GSC availability on the March 2026 partition

query2 = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_true,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS FALSE
    ) AS gsc_available_false,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS NULL
    ) AS gsc_available_null
FROM read_parquet('{march_path}')
"""

result2 = con.execute(query2).fetchdf()
display(result2)

,total_rows,gsc_available_true,gsc_available_false,gsc_available_null
0,9841378,3611061,6230317,0


In [65]:
# Query 3: Count rows surviving the explicit GSC availability filter

query3 = f"""
SELECT
    COUNT(*) AS rows_after_gsc_filter
FROM read_parquet('{march_path}')
WHERE gsc_data_available IS TRUE
"""

result3 = con.execute(query3).fetchdf()
display(result3)

,rows_after_gsc_filter
0,3611061


## 4. Feature set and prediction-time availability

The prediction unit is one client-content pair on a report date. The target is the next day's GSC clicks (`label_next_day_clicks`).

The initial feature set uses three previous-day GSC metrics:

- `prev_day_clicks`: clicks recorded on the previous calendar day. This is historical performance available before the prediction date.
- `prev_day_impressions`: impressions recorded on the previous calendar day. This provides a historical measure of search visibility.
- `prev_day_avg_position`: average search position recorded on the previous calendar day. This provides historical ranking context; missing values must be handled explicitly if they occur in a different sample.

The construction requires consecutive calendar dates and `gsc_data_available IS TRUE` for the previous, prediction, and next-day records. The current-day and next-day performance metrics are not included as features. The next-day clicks field is the label, not a feature.

These are candidate features for a baseline experiment. Their use assumes the previous day's finalized metrics would be available at the intended prediction time; actual production data latency must be verified before deployment.

## 5. Deliberate leakage experiment

I trained a Random Forest regression baseline using only previous-day GSC clicks, impressions, and average position. On a reproducible random 80/20 train-test split of a sample of up to 100,000 eligible rows, the baseline achieved MAE 0.2983 and R² 0.4954.

I then deliberately introduced `label_next_day_clicks` into the feature matrix while keeping it as the target. This invalid experiment produced MAE 0.0058 and R² 0.8632. The apparent improvement is caused by target leakage: the model receives the answer it is supposed to predict.

The leaked score must not be reported as valid model performance. The label-derived feature is removed from the production candidate feature set, and the honest baseline results are retained only as an initial benchmark. The random split may also place observations from the same client-content pair in both sets, so these metrics are preliminary rather than proof of generalization to unseen clients or future dates.

## 6. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The inspected daily performance sample covers only June 2026, so it cannot establish long-term trends, seasonality, or future performance. Data availability is uneven: GSC metrics are unavailable for many rows, and GA4 availability is also incomplete. Missing metrics must not automatically be interpreted as zero activity. The query-performance table covers a fixed 90-day window from 2 April to 30 June 2026, which overlaps the June daily sample and has a different unit of analysis. Combining these tables requires careful date alignment, join-cardinality checks, and leakage prevention. The observational data can support descriptive analysis and cautious decision-support recommendations, but cannot by itself establish that a particular change caused an improvement in search performance.

In [66]:

import pandas as pd
import duckdb

# Confirm the March partition is available
assert "march_path" in globals(), "Run the March download cell first."

con = duckdb.connect()

# Inspect the fields needed for next-day target construction
schema = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{march_path}')"
).fetchdf()

print("Required columns:")
required = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_data_available",
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position",
]
available = schema["column_name"].tolist()

for col in required:
    print(f"{col}: {'OK' if col in available else 'MISSING'}")

assert all(col in available for col in required), \
    "A required column is missing. Check the schema output."


Required columns:
report_date: OK
client_hash_id: OK
content_hash_id: OK
gsc_data_available: OK
gsc_clicks: OK
gsc_impressions: OK
gsc_avg_position: OK


In [67]:

query = f"""
WITH daily AS (
    SELECT
        CAST(report_date AS DATE) AS report_date,
        client_hash_id,
        content_hash_id,
        gsc_data_available,
        gsc_clicks,
        gsc_impressions,
        gsc_avg_position
    FROM read_parquet('{march_path}')
),
aligned AS (
    SELECT
        *,
        LAG(report_date) OVER w AS prev_date,
        LAG(gsc_data_available) OVER w AS prev_gsc_available,
        LAG(gsc_clicks) OVER w AS prev_day_clicks,
        LAG(gsc_impressions) OVER w AS prev_day_impressions,
        LAG(gsc_avg_position) OVER w AS prev_day_avg_position,
        LEAD(report_date) OVER w AS next_date,
        LEAD(gsc_data_available) OVER w AS next_gsc_available,
        LEAD(gsc_clicks) OVER w AS next_day_clicks
    FROM daily
    WINDOW w AS (
        PARTITION BY client_hash_id, content_hash_id
        ORDER BY report_date
    )
)
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    prev_day_clicks,
    prev_day_impressions,
    prev_day_avg_position,
    next_day_clicks AS label_next_day_clicks
FROM aligned
WHERE prev_date = report_date - INTERVAL '1 day'
  AND next_date = report_date + INTERVAL '1 day'
  AND prev_gsc_available IS TRUE
  AND gsc_data_available IS TRUE
  AND next_gsc_available IS TRUE
"""

model_df = con.execute(query).fetchdf()

print("Eligible rows:", len(model_df))
print("Date range:", model_df["report_date"].min(), "to",
      model_df["report_date"].max())
print("Missing feature/label values:")
print(model_df[
    ["prev_day_clicks", "prev_day_impressions",
     "prev_day_avg_position", "label_next_day_clicks"]
].isna().sum())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible rows: 2936405
Date range: 2026-03-02 00:00:00 to 2026-03-30 00:00:00
Missing feature/label values:
prev_day_clicks          0
prev_day_impressions     0
prev_day_avg_position    0
label_next_day_clicks    0
dtype: int64


In [68]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import pandas as pd

# Use a reproducible sample to keep training practical
sample_df = model_df.sample(
    n=min(100_000, len(model_df)),
    random_state=42
).copy()

feature_cols = [
    "prev_day_clicks",
    "prev_day_impressions",
    "prev_day_avg_position",
]
label_col = "label_next_day_clicks"

X = sample_df[feature_cols]
y = sample_df[label_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Honest baseline: only previous-day features
baseline_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

print("HONEST BASELINE")
print("MAE:", round(mean_absolute_error(y_test, baseline_pred), 4))
print("R²:", round(r2_score(y_test, baseline_pred), 4))

# Deliberately leaked model: label accidentally included as a feature
X_leaky = sample_df[feature_cols + [label_col]]
Xl_train, Xl_test, yl_train, yl_test = train_test_split(
    X_leaky, y, test_size=0.2, random_state=42
)

leaky_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
leaky_model.fit(Xl_train, yl_train)
leaky_pred = leaky_model.predict(Xl_test)

print("\nINTENTIONALLY LEAKED MODEL (INVALID)")
print("MAE:", round(mean_absolute_error(yl_test, leaky_pred), 4))
print("R²:", round(r2_score(yl_test, leaky_pred), 4))


HONEST BASELINE
MAE: 0.2923
R²: 0.5397

INTENTIONALLY LEAKED MODEL (INVALID)
MAE: 0.0005
R²: 0.9988


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.